# KökLM × Llama 3.2: gerçek ağırlıklarla çalıştır

KökLM'in mimarisi Llama 3 ile birebir uyumlu. Bu not defteri Meta'nın **Llama 3.2 1B Instruct** ağırlıklarını indirir, KökLM'e yükler, Hugging Face'in kendi Llama uygulamasıyla karşılaştırır ve modelle sohbet etmeni sağlar.

**Başlamadan önce:**
1. Üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
2. [huggingface.co/meta-llama/Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) sayfasında Meta'nın lisansını kabul et (onay genelde birkaç dakika sürer).
3. [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) adresinden bir **Read** token oluştur.

In [ ]:
# Kaç kez çalıştırılırsa çalıştırılsın aynı yere kurar
import os, sys, subprocess
os.chdir("/content")
if not os.path.isdir("/content/koklm/.git"):
    subprocess.run(["rm", "-rf", "/content/koklm"])
    subprocess.run(["git", "clone", "-q", "https://github.com/mkor4y/koklm", "/content/koklm"], check=True)
else:
    subprocess.run(["git", "-C", "/content/koklm", "pull", "-q"], check=True)
os.chdir("/content/koklm")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[hf]"], check=True)
if "/content/koklm" not in sys.path:
    sys.path.insert(0, "/content/koklm")
print("hazır:", os.getcwd())

## 1. Hugging Face girişi
Token'ını yapıştır.

In [ ]:
from huggingface_hub import login
login()

## 2. Llama ağırlıklarını KökLM'e dönüştür ve doğrula
Ağırlıklar (~2,5 GB) iner, KökLM checkpoint'ine çevrilir ve aynı cümle üzerinde Hugging Face Llama'sıyla karşılaştırılır.

In [ ]:
!cd /content/koklm && python scripts/llama_donustur.py --hf meta-llama/Llama-3.2-1B-Instruct \
    --cikti checkpoints/llama32_1b_instruct.pt --dtype float16 --dogrula

## 3. KökLM ile sohbet

In [ ]:
import torch
from koklm.yukle import checkpoint_yukle
from koklm.tokenizer import tokenizer_al

dev = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if dev == "cuda" else torch.float32
model, ck = checkpoint_yukle("/content/koklm/checkpoints/llama32_1b_instruct.pt", dtype=dtype, device=dev)
tok = tokenizer_al(ck["veri_meta"]["tokenizer"])
print(f"{model.parametre_sayisi():,} parametre yüklendi ({dev})")

gecmis = [{"role": "system", "content": "Sen yardımsever bir asistansın. Türkçe cevap ver."}]

def sor(soru, max_new_tokens=256, temperature=0.7):
    gecmis.append({"role": "user", "content": soru})
    ids = tok.sohbet_kodla(gecmis)
    x = torch.tensor([ids], device=dev)
    out = model.generate(x, max_new_tokens, temperature, 0.9, tok.dur_ids, vocab_siniri=tok.vocab_size)
    cevap = tok.decode(out[0, len(ids):].tolist()).strip()
    gecmis.append({"role": "assistant", "content": cevap})
    print(cevap)

In [ ]:
sor("Transformer mimarisinde attention ne işe yarar? Kısaca anlat.")

In [ ]:
sor("Peki RoPE nedir?")

## 4. Bir bloğun içine bak
Gerçek ağırlıklar artık KökLM'in kendi katmanlarında. Örneğin ilk bloğun matris boyutları:

In [ ]:
b = model.layers[0]
for ad, p in b.named_parameters():
    print(f"{ad:<28} {tuple(p.shape)}  {p.numel():>12,}")

## 5. Türkçe ince ayar (isteğe bağlı)
Llama ağırlıklarından başlayıp kendi Türkçe metinlerinle eğitebilirsin:

```bash
python scripts/prepare_data.py --girdi metinler/ --cikti data/tr --tokenizer hf:meta-llama/Llama-3.2-1B-Instruct
python scripts/train.py --baslangic checkpoints/llama32_1b_instruct.pt --data data/tr \
    --seq_len 1024 --batch_size 4 --grad_accum 8 --lr 2e-5 --min_lr 2e-6 --warmup 50 \
    --max_steps 1000 --grad_checkpoint
```

Not: 1B modelin tam ince ayarı AdamW ile ~18 GiB durum belleği ister; T4'ün 16 GB'ı yetmez. Bunun için A100 gibi daha büyük bir GPU gerekir (LoRA desteği yol haritasında).